![Health-Informatics: A Python Tutorial](../../Image/chapter-banner.png)


# Module 9: Privacy, De-identification, and Governance



**Health Informatics in Python** · Part III: Privacy, Text & Applied Informatics · Module 9 of 16

---



Health data is among the most sensitive data there is, and its use is governed by
law. Before data can be shared for research or analytics, **protected health
information (PHI)** usually has to be removed. This module covers the **HIPAA**
de-identification standards and implements de-identification for both structured
tables and free text.


## Learning objectives

By the end of this module you will be able to:

1. Distinguish HIPAA **Safe Harbor** (the 18 identifiers) from **Expert Determination**.
2. De-identify **structured** data by suppressing, generalizing, and hashing identifiers.
3. De-identify **free text** using regex plus spaCy named-entity recognition.
4. Measure **k-anonymity** over quasi-identifiers and generalize to reach a target *k*.
5. Explain **synthetic data** and governance controls (IRB, DUA, minimum necessary).

> **Ethics note.** All names, MRNs, dates, and contacts in this notebook are
> **fabricated**. De-identification is safety-critical: the methods here are
> educational, and production systems should use validated tooling and expert review.


## Dataset

The synthetic EHR plus two **fabricated** clinical notes seeded with PHI-shaped
tokens (names, MRNs, dates, phone, email, address) so the de-identification code
has realistic targets.


In [6]:
# --- Self-contained synthetic EHR generator (identical to Parts I-II) ---
import numpy as np
import pandas as pd

def make_synthetic_ehr(n_patients=200, seed=42):
    rng = np.random.default_rng(seed)
    first = ["Ava","Liam","Noah","Mia","Zoe","Omar","Ivan","Sara","Leo","Nina",
             "Ruth","Kai","Yara","Theo","Ida","Sam","Ana","Eli","Rex","Uma"]
    last  = ["Khan","Ortiz","Chen","Diaz","Patel","Ali","Brown","Nash","Reed","Vega",
             "Cole","Frost","Grant","Hale","Iqbal","Jain","Kerr","Lund","Mora","Park"]
    ages  = rng.integers(18, 90, size=n_patients)
    patients = pd.DataFrame({
        "patient_id": [f"P{1000+i}" for i in range(n_patients)],
        "given_name": rng.choice(first, size=n_patients),
        "family_name": rng.choice(last, size=n_patients),
        "sex": rng.choice(["male","female"], size=n_patients, p=[0.49,0.51]),
        "age": ages, "birth_year": 2026 - ages,
    })
    enc_rows, enc_types = [], ["ambulatory","emergency","inpatient","wellness"]
    for pid in patients["patient_id"]:
        for _ in range(rng.integers(1, 5)):
            day = rng.integers(0, 365*3)
            enc_rows.append({"encounter_id": f"E{len(enc_rows)+1:05d}", "patient_id": pid,
                "encounter_type": rng.choice(enc_types, p=[0.55,0.15,0.10,0.20]),
                "date": (pd.Timestamp("2023-01-01") + pd.Timedelta(days=int(day))).date()})
    encounters = pd.DataFrame(enc_rows)
    obs_defs = [("Body height","cm",150,195),("Body weight","kg",50,110),
                ("Systolic blood pressure","mmHg",100,165),("Heart rate","/min",55,100),
                ("Hemoglobin A1c","%",4.8,9.5)]
    obs_rows = []
    for _, e in encounters.iterrows():
        for name, unit, lo, hi in obs_defs:
            if rng.random() < 0.7:
                obs_rows.append({"observation_id": f"O{len(obs_rows)+1:06d}",
                    "encounter_id": e["encounter_id"], "patient_id": e["patient_id"],
                    "observation": name, "value": round(float(rng.uniform(lo,hi)),1),
                    "unit": unit, "date": e["date"]})
    observations = pd.DataFrame(obs_rows)
    cond_pool = ["Essential hypertension","Type 2 diabetes mellitus","Asthma",
                 "Acute bronchitis","Major depressive disorder","Osteoarthritis",
                 "Chronic kidney disease","Anemia"]
    cond_rows = []
    for pid in patients["patient_id"]:
        for c in rng.choice(cond_pool, size=rng.integers(0,4), replace=False):
            cond_rows.append({"condition_id": f"C{len(cond_rows)+1:05d}",
                              "patient_id": pid, "condition": c})
    conditions = pd.DataFrame(cond_rows)
    med_pool = ["Lisinopril","Metformin","Albuterol","Atorvastatin","Sertraline",
                "Amoxicillin","Ibuprofen","Hydrochlorothiazide"]
    med_rows = []
    for pid in patients["patient_id"]:
        for m in rng.choice(med_pool, size=rng.integers(0,4), replace=False):
            med_rows.append({"medication_id": f"M{len(med_rows)+1:05d}",
                             "patient_id": pid, "medication": m})
    medications = pd.DataFrame(med_rows)
    return {"patients":patients,"encounters":encounters,"observations":observations,
            "conditions":conditions,"medications":medications}

ehr = make_synthetic_ehr()
print("Tables:", ", ".join(f"{k} ({len(v)})" for k,v in ehr.items()))


Tables: patients (200), encounters (511), observations (1804), conditions (304), medications (276)


In [7]:
# Synthetic clinical notes containing FABRICATED identifiers (no real people)
notes = [
    {"note_id":"N001","patient_id":"P1000","text":
     "Ava Khan (MRN 4471982) seen on 03/12/2024. Contact 607-555-0148. "
     "58 yo F with h/o hypertension and type 2 diabetes presents for follow-up. "
     "BP 148/88. Denies chest pain. No shortness of breath. Continue lisinopril; "
     "recheck A1c in 3 months. Lives at 12 Elm St, Ithaca NY."},
    {"note_id":"N002","patient_id":"P1001","text":
     "Liam Ortiz, DOB 07/22/1969, MRN 5580321. Presents with acute bronchitis. "
     "Reports cough x5 days. Negative for fever. Family history of asthma. "
     "Started amoxicillin. Follow up if not improving. Email liam.o@example.com."},
]
for n in notes:
    print(n["note_id"], "-", n["text"][:70], "...")


N001 - Ava Khan (MRN 4471982) seen on 03/12/2024. Contact 607-555-0148. 58 yo ...
N002 - Liam Ortiz, DOB 07/22/1969, MRN 5580321. Presents with acute bronchiti ...


## 9.1 The two HIPAA de-identification standards

| Standard              | How it works                                                                | Trade-off                                      |
|-----------------------|-----------------------------------------------------------------------------|------------------------------------------------|
| **Safe Harbor**       | Remove **18 specific identifiers** named by HIPAA regulations from all data. | Straightforward, rule-based; may remove useful data along with identifiers. |
| **Expert Determination** | An expert assesses and documents that the risk of re-identification is "very small," allowing custom methods to reduce such risk. | Allows more data utility and context sensitivity; requires specialized knowledge and thorough documentation. |


**Safe Harbor:**  
Relies on a strict list of 18 personal identifiers (such as names, addresses smaller than state, full dates, and contact info) that must be removed to consider data de-identified. This method is relatively easy to apply and does not require specialized statistical or privacy expertise. However, it can inadvertently remove more information than strictly necessary for privacy, potentially harming research utility.

**Expert Determination:**  
Involves hiring or relying on a qualified expert in statistical and scientific methods who will identify direct and indirect identifiers in the dataset and certify that the likelihood of someone being re-identified from the data is very small. This approach allows for nuanced balancing between privacy and information loss, accommodating unique data situations, but it is more complex and requires expert oversight and documentation.

**18 Safe Harbor identifiers** (must be removed or generalized): names, any small-area geographic indicator (smaller than a state, e.g. city, zip code except for the initial 3 digits in some cases), all dates directly related to an individual (other than year), ages over 89, phone/fax numbers, emails, social security numbers, medical record/account/license numbers, vehicle/device identifiers, URLs, IP addresses, biometric identifiers, full-face photographs/images, and any other unique identifying code or characteristic.


In [8]:
SAFE_HARBOR_18 = [
    "Names","Geographic subdivisions < state","Dates (finer than year); ages > 89",
    "Telephone numbers","Fax numbers","Email addresses","Social Security numbers",
    "Medical record numbers","Health plan beneficiary numbers","Account numbers",
    "Certificate/license numbers","Vehicle identifiers","Device identifiers",
    "Web URLs","IP addresses","Biometric identifiers","Full-face photos",
    "Any other unique identifying number/characteristic/code",
]
print(f"HIPAA Safe Harbor identifiers ({len(SAFE_HARBOR_18)}):")
for i, ident in enumerate(SAFE_HARBOR_18, 1):
    print(f"{i:2d}. {ident}")

HIPAA Safe Harbor identifiers (18):
 1. Names
 2. Geographic subdivisions < state
 3. Dates (finer than year); ages > 89
 4. Telephone numbers
 5. Fax numbers
 6. Email addresses
 7. Social Security numbers
 8. Medical record numbers
 9. Health plan beneficiary numbers
10. Account numbers
11. Certificate/license numbers
12. Vehicle identifiers
13. Device identifiers
14. Web URLs
15. IP addresses
16. Biometric identifiers
17. Full-face photos
18. Any other unique identifying number/characteristic/code


## 9.2 De-identifying structured data

When working with structured (tabular) data, removing identifiers is a column-by-column
process. Each column containing an identifier should have an appropriate de-identification
strategy assigned to it. Common strategies include:
- **Suppress** (drop): Remove the column entirely if it directly identifies a person (e.g., name).
- **Generalize** (reduce precision): Broaden the information so it's less specific, such as
  reporting age groups instead of exact ages, or using birth year instead of full birth date.
- **Pseudonymize** (hash to a consistent token): Replace an identifier, like a patient ID, with
  a generated code (using a hash function and possibly a secret salt) that is stable within
  the dataset but cannot be used to recover the original value. 
This approach balances privacy protection with the need to retain useful, analyzable data.


In [9]:
import hashlib
patients = ehr["patients"].copy()

def pseudonymize(value, salt="study-2026"):
    return "PT-" + hashlib.sha256((salt+str(value)).encode()).hexdigest()[:10]

deid = pd.DataFrame({
    # pseudonymize the direct identifier (stable across the dataset, not reversible without salt)
    "study_id": patients["patient_id"].map(pseudonymize),
    # suppress names entirely
    # generalize: age > 89 collapses to '90+' per Safe Harbor; keep birth YEAR only
    "age_group": pd.cut(patients["age"], [0,18,40,65,89,200],
                        labels=["<18","18-39","40-64","65-89","90+"]),
    "sex": patients["sex"],
})
print("De-identified structured table (names suppressed, id hashed, age generalized):")
deid.head()

De-identified structured table (names suppressed, id hashed, age generalized):


,study_id,age_group,sex
0,PT-4e1ad65a22,18-39,female
1,PT-29aad3ddf2,65-89,male
2,PT-5a1b9690e7,40-64,female
3,PT-99115d1b96,40-64,female
4,PT-cabe50e931,40-64,female


## 9.3 De-identifying free text

De-identifying free text is more complex than structured data because personal identifiers
may appear anywhere within unstructured prose. To address this, we use a combination of:
- **Regular expressions (regex):** These can reliably detect well-defined patterns such as
  Medical Record Numbers (MRNs), phone numbers, emails, dates, and similar structured strings.
- **spaCy Named Entity Recognition (NER):** This tool detects names of people and places
  (like "John Smith" or "New York") in natural language, even if their format is less predictable.

For maximum portability, the code attempts to load the spaCy NER model, but if it is not
available (e.g., on a new system), the notebook will still run, simply skipping the NER step.


In [10]:
import re
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    NER = True
except Exception:
    nlp = None
    NER = False
print("spaCy NER model available:", NER)

PATTERNS = {
    "MRN":   re.compile(r"\bMRN\s*\d{5,}\b", re.I),
    "PHONE": re.compile(r"\b\d{3}-\d{3}-\d{4}\b"),
    "EMAIL": re.compile(r"\b[\w.]+@[\w.]+\.\w+\b"),
    "DATE":  re.compile(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b"),
    "SSN":   re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
}

def deidentify_text(text):
    redactions = []
    # 1) regex-based structured identifiers
    for tag, pat in PATTERNS.items():
        for m in pat.finditer(text):
            redactions.append((m.start(), m.end(), tag))
    # 2) NER-based names and places
    if NER:
        for ent in nlp(text).ents:
            if ent.label_ in {"PERSON","GPE","LOC","FAC"}:
                redactions.append((ent.start_char, ent.end_char, ent.label_))
    # apply right-to-left so offsets stay valid
    out = text
    for start, end, tag in sorted(redactions, key=lambda r: r[0], reverse=True):
        out = out[:start] + f"[{tag}]" + out[end:]
    return out

spaCy NER model available: True


### Milestone 1 - redact a clinical note

In [11]:
original = notes[0]["text"]
redacted = deidentify_text(original)
print("ORIGINAL:\n", original)
print("\nDE-IDENTIFIED:\n", redacted)

ORIGINAL:
 Ava Khan (MRN 4471982) seen on 03/12/2024. Contact 607-555-0148. 58 yo F with h/o hypertension and type 2 diabetes presents for follow-up. BP 148/88. Denies chest pain. No shortness of breath. Continue lisinopril; recheck A1c in 3 months. Lives at 12 Elm St, Ithaca NY.

DE-IDENTIFIED:
 [PERSON] ([MRN]) seen on [DATE]. Contact [PHONE]. 58 yo F with h/o hypertension and type 2 diabetes presents for follow-up. BP 148/88. Denies chest pain. No shortness of breath. Continue lisinopril; recheck A1c in 3 months. Lives at 12 [FAC].


In [12]:
# quick check: did the obvious identifiers get masked?
for token in ["4471982","607-555-0148","03/12/2024","Ava","Ithaca"]:
    print(f"  {token:14s} redacted: {token not in redacted}")

  4471982        redacted: True
  607-555-0148   redacted: True
  03/12/2024     redacted: True
  Ava            redacted: True
  Ithaca         redacted: True


## 9.4 k-anonymity

**k-anonymity** is a privacy concept that addresses the risk of re-identifying individuals in a dataset, even after obvious identifiers (like names or SSNs) are removed. It recognizes that combinations of seemingly harmless features—such as age, sex, or ZIP code—can still uniquely identify people when combined. These features are known as **quasi-identifiers**.

A dataset is considered **k-anonymous** if every unique combination of quasi-identifiers appears in at least *k* different records. This means each individual is hidden in a crowd of at least *k* people sharing the same quasi-identifier values. The higher the value of *k*, the stronger the privacy guarantee: it’s harder to pick out an individual from the group.

Achieving k-anonymity often requires generalizing or coarsening the data (for example, grouping ages into ranges, or truncating ZIP codes) so that more records look alike, increasing the minimum group size. The following sections show how to measure and improve k-anonymity using real and generalized data.

Removing direct identifiers isn't enough: **quasi-identifiers** (age, sex, ZIP) can
re-identify people by combination. A dataset is **k-anonymous** if every combination
of quasi-identifiers is shared by at least *k* records. We measure it, then
generalize to raise the minimum *k*.


In [13]:
rng = np.random.default_rng(1)
qi = ehr["patients"][["age","sex"]].copy()
qi["zip3"] = rng.choice(["148","149","130"], size=len(qi))   # coarse ZIP (first 3)

def min_k(df, cols):
    return df.groupby(cols).size().min()

print("Using exact age as a quasi-identifier:")
print("  smallest group size (k):", min_k(qi, ["age","sex","zip3"]))

# generalize age into bands -> larger groups -> higher k
qi["age_band"] = pd.cut(qi["age"], [0,40,65,120], labels=["<40","40-64","65+"])
k2 = min_k(qi, ["age_band","sex","zip3"])
print("After generalizing age into bands:")
print("  smallest group size (k):", k2, "->", "meets k>=5" if k2>=5 else "still too small")

Using exact age as a quasi-identifier:
  smallest group size (k): 1
After generalizing age into bands:
  smallest group size (k): 6 -> meets k>=5


/tmp/ipykernel_64707/1139078976.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return df.groupby(cols).size().min()


## 9.5 Synthetic data and governance

The safest way to *share* is often not to share real records at all but a
**synthetic** dataset that preserves statistical structure without mapping to real
people — exactly what this series' generator does.

Beyond technical methods, health data use sits inside a **governance** framework:

- **IRB approval** for human-subjects research
- **Data Use Agreements (DUAs)** governing what a recipient may do
- **Minimum necessary** — access only the data required for the purpose
- **Limited Data Sets** — a middle ground retaining dates/geography under a DUA

Production de-identification tools worth knowing: **Microsoft Presidio**, **Philter**,
and **NLM Scrubber**. Formal privacy guarantees come from **differential privacy**.


In [14]:
# The generator IS the privacy strategy: no row maps to a real person.
synth = make_synthetic_ehr(n_patients=5, seed=99)["patients"]
print("Synthetic patients — statistically shaped, entirely fabricated:")
print(synth[["patient_id","given_name","family_name","sex","age"]].to_string(index=False))

Synthetic patients — statistically shaped, entirely fabricated:
patient_id given_name family_name    sex  age
     P1000       Ruth       Brown   male   86
     P1001        Rex       Frost female   54
     P1002        Uma       Grant female   72
     P1003        Uma         Ali   male   58
     P1004       Yara        Park   male   30


## Exercises

1. Add a **street-address** regex to `PATTERNS` and confirm the address in note N001
   is redacted without relying on NER.
2. De-identify **note N002** and verify the email and DOB are masked.
3. Compute k-anonymity treating **exact ZIP5** as a quasi-identifier, then show which
   generalization (age band vs ZIP3) buys more anonymity.



## Key takeaways

- HIPAA offers two paths: **Safe Harbor** (remove 18 identifiers) and **Expert Determination**.
- Structured de-id = **suppress / generalize / pseudonymize**; free-text de-id needs
  **regex + NER** together.
- **k-anonymity** guards against quasi-identifier re-identification.
- **Synthetic data** and **governance** (IRB, DUA, minimum necessary) complete the picture.



---
*Next: Module 10 - Clinical Natural Language Processing.*
